Install Dependencies

Install Litellm to enable the use of any vendor models

In [10]:
# Install ADK and LiteLLM
!pip install google-adk -q
!pip install litellm -q
!pip install google-cloud-aiplatform[agent_engines,adk]>=1.112 -U -q

print("Dependencies installed successfully...")

Dependencies installed successfully...


Configure environment

In [102]:
import os
from getpass import getpass

# Get inputs
PROJECT_ID = getpass("Enter your GCP project id: ")
GOOGLE_MAPS_API_KEY = getpass("Enter your Google Maps API key: ")
GEMINI_API_KEY = getpass("Enter your Google Gemini API key: ")
LOCATION = "us-central1"

# Set environment variables so LiteLLM and your functions automatically find them
os.environ["GOOGLE_MAPS_API_KEY"] = GOOGLE_MAPS_API_KEY
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["LOCATION"] = LOCATION

print("Credentials loaded successfully into environment!")

Enter your GCP project id: ··········
Enter your Google Maps API key: ··········
Enter your Google Gemini API key: ··········
Credentials loaded successfully into environment!


Create a storage bucket to host the files for deployment

In [103]:
BUCKET_NAME="challenge-5-bc"

!gcloud storage buckets create gs://$BUCKET_NAME \
    --project={PROJECT_ID} \
    --location={LOCATION}

Creating gs://challenge-5-bc/...
ERROR: (gcloud.storage.buckets.create) HTTPError 409: Your previous request to create the named bucket succeeded and you already own it.


In [105]:
vertexai.init(
    project=PROJECT_ID,
    location="global",
)

In [106]:
# Import Agent Platform and initialize the SDK

import vertexai

client = vertexai.Client(
    project=PROJECT_ID,
    location=LOCATION,
)

Get NWS forecast based on coordinates

In [107]:
from typing import Any, Dict, List, Union
import pandas as pd
import requests


def get_forecast_by_coordinates(
    latitude: float,
    longitude: float,
    user_agent: str = "GoogleColabNotebook/1.0 (user@example.com)",
    return_dataframe: bool = True,
    timeout: int = 10,
) -> Union[pd.DataFrame, List[Dict[str, Any]]]:
    """Retrieve weather forecast data from the NWS API for specific coordinates.

    Executes the two-stage NWS lookup process:
    1. Resolves grid points from `https://api.weather.gov/points/{lat},{lon}`.
    2. Retrieves period forecasts from the point's `forecast` property endpoint.

    Args:
        latitude (float): Latitude in decimal degrees (-90.0 to 90.0).
        longitude (float): Longitude in decimal degrees (-180.0 to 180.0).
        user_agent (str): Identification header required by NWS guidelines.
            Defaults to 'GoogleColabNotebook/1.0 (user@example.com)'.
        return_dataframe (bool): If True, returns a pandas DataFrame for neat
            display in Colab. If False, returns raw dictionary periods.
            Defaults to True.
        timeout (int): HTTP request timeout in seconds. Defaults to 10.

    Returns:
        Union[pd.DataFrame, List[Dict[str, Any]]]: A pandas DataFrame containing
            forecast periods if `return_dataframe=True`, or a raw list of
            dictionaries if False.

    Raises:
        ValueError: If coordinates are out of valid geographic ranges.
        requests.exceptions.HTTPError: If NWS API requests fail.
        requests.exceptions.RequestException: For network or connection errors.
    """
    # Validate coordinate ranges
    if not (-90.0 <= latitude <= 90.0):
        raise ValueError(f"Latitude must be between -90 and 90 degrees. Got {latitude}.")
    if not (-180.0 <= longitude <= 180.0):
        raise ValueError(f"Longitude must be between -180 and 180 degrees. Got {longitude}.")

    # Cap precision to 4 decimal places per NWS API recommendations
    lat_str = f"{latitude:.4f}"
    lon_str = f"{longitude:.4f}"

    headers = {
        "User-Agent": user_agent,
        "Accept": "application/geo+json",
    }

    # Step 1: Query points endpoint to get metadata and grid info
    points_url = f"https://api.weather.gov/points/{lat_str},{lon_str}"
    points_response = requests.get(points_url, headers=headers, timeout=timeout)
    points_response.raise_for_status()

    points_data = points_response.json()
    forecast_url = points_data.get("properties", {}).get("forecast")

    if not forecast_url:
        raise KeyError("Grid metadata response did not contain a valid 'forecast' URL.")

    # Step 2: Query the grid forecast endpoint
    forecast_response = requests.get(forecast_url, headers=headers, timeout=timeout)
    forecast_response.raise_for_status()

    forecast_data = forecast_response.json()
    periods: List[Dict[str, Any]] = forecast_data.get("properties", {}).get("periods", [])

    if return_dataframe:
        df = pd.DataFrame(periods)
        # Reorder key columns to the front for better visibility in Colab
        preferred_cols = ["name", "temperature", "temperatureUnit", "windSpeed", "windDirection", "shortForecast"]
        existing_cols = [col for col in preferred_cols if col in df.columns]
        other_cols = [col for col in df.columns if col not in existing_cols]

        return df[existing_cols + other_cols]

    return periods

In [108]:
from typing import List, Dict, Any


def get_weather_forecast(
    latitude: float,
    longitude: float,
) -> List[Dict[str, Any]]:
    """
    Fetches the US National Weather Service forecast for coordinates.

    Args:
        latitude: Latitude coordinate.
        longitude: Longitude coordinate.

    Returns:
        A list of forecast periods containing weather information.
    """

    return get_forecast_by_coordinates(
        latitude=latitude,
        longitude=longitude,
        return_dataframe=False,
    )

In [109]:
# test get_forecast_by_coordinates function

# Fetch forecast as a Pandas DataFrame
df_forecast = get_forecast_by_coordinates(
    latitude=39.7456,
    longitude=-97.0892,
    user_agent="MyColabExperiment/1.0 (myemail@example.com)"
)

# Display table in Google Colab
df_forecast.head()

,name,temperature,temperatureUnit,windSpeed,windDirection,shortForecast,number,startTime,endTime,isDaytime,temperatureTrend,probabilityOfPrecipitation,icon,detailedForecast
0,This Afternoon,89,F,5 to 15 mph,SW,Sunny,1,2026-08-07T12:00:00-05:00,2026-08-07T18:00:00-05:00,True,None,"{'unitCode': 'wmoUnit:percent', 'value': 4}",https://api.weather.gov/icons/land/day/few?siz...,"Sunny, with a high near 89. Southwest wind 5 t..."
1,Tonight,68,F,5 mph,SE,Mostly Clear,2,2026-08-07T18:00:00-05:00,2026-08-08T06:00:00-05:00,False,None,"{'unitCode': 'wmoUnit:percent', 'value': 2}",https://api.weather.gov/icons/land/night/few?s...,"Mostly clear, with a low around 68. Southeast ..."
2,Saturday,89,F,5 to 15 mph,SE,Sunny,3,2026-08-08T06:00:00-05:00,2026-08-08T18:00:00-05:00,True,None,"{'unitCode': 'wmoUnit:percent', 'value': 3}",https://api.weather.gov/icons/land/day/few?siz...,"Sunny, with a high near 89. Southeast wind 5 t..."
3,Saturday Night,73,F,10 to 15 mph,S,Mostly Clear,4,2026-08-08T18:00:00-05:00,2026-08-09T06:00:00-05:00,False,None,"{'unitCode': 'wmoUnit:percent', 'value': 3}",https://api.weather.gov/icons/land/night/few?s...,"Mostly clear, with a low around 73. South wind..."
4,Sunday,93,F,10 to 15 mph,S,Mostly Sunny,5,2026-08-09T06:00:00-05:00,2026-08-09T18:00:00-05:00,True,None,"{'unitCode': 'wmoUnit:percent', 'value': 8}",https://api.weather.gov/icons/land/day/sct?siz...,"Mostly sunny, with a high near 93. South wind ..."


Function to get the lat/lon coordinates based on the city and state

In [110]:
import os
from typing import Tuple
import requests


def get_coordinates(
    city: str,
    state: str,
    api_key: str = None,
    timeout: int = 10,
) -> Tuple[float, float]:
    """Convert a city and state into geographic latitude and longitude coordinates.

    Uses the Google Maps Geocoding API to resolve address strings to spatial coordinates.

    Args:
        city (str): The name of the city (e.g., 'Austin', 'Seattle').
        state (str): The state name or 2-letter postal abbreviation (e.g., 'TX', 'Washington').
        api_key (str, optional): Google Maps API Key. If not provided, reads from
            the 'GOOGLE_MAPS_API_KEY' environment variable.
        timeout (int): Request timeout in seconds. Defaults to 10.

    Returns:
        Tuple[float, float]: A tuple containing (latitude, longitude) as floats.

    Raises:
        ValueError: If an API key is missing or no results are returned for the input.
        requests.exceptions.HTTPError: If the Google API request fails.
    """
    key = api_key or os.getenv("GOOGLE_MAPS_API_KEY")
    if not key:
        raise ValueError(
            "Google Maps API Key required. Pass 'api_key' argument or set GOOGLE_MAPS_API_KEY env var."
        )

    address_str = f"{city.strip()}, {state.strip()}"
    base_url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {
        "address": address_str,
        "key": key,
    }

    response = requests.get(base_url, params=params, timeout=timeout)
    response.raise_for_status()

    data = response.json()
    status = data.get("status")

    if status != "OK" or not data.get("results"):
        error_msg = data.get("error_message", f"Geocoding API status: {status}")
        raise ValueError(f"Could not resolve coordinates for '{address_str}'. {error_msg}")

    location = data["results"][0]["geometry"]["location"]
    return location["lat"], location["lng"]

In [111]:
from typing import Dict


def lookup_coordinates(
    city: str,
    state: str,
) -> Dict[str, float]:
    """
    Converts a US city and state into latitude and longitude coordinates.

    Args:
        city: US city name.
        state: US state abbreviation or name.

    Returns:
        Dictionary containing latitude and longitude.
    """

    latitude, longitude = get_coordinates(
        city=city,
        state=state,
    )

    return {
        "latitude": latitude,
        "longitude": longitude,
    }

In [112]:
# test get_coordinates function
print(get_coordinates(city="Pittsburgh", state="PA", api_key=GOOGLE_MAPS_API_KEY))

(40.4386612, -79.99723519999999)


In [137]:
import os
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part

# 1. System Instruction defining the agent's boundaries
SYSTEM_PROMPT = """
You are a helpful assistant specialized strictly in US weather forecasts using the National Weather Service (NWS).

Guidelines:
1. Scope: You only answer weather-related queries and questions about locations within the United States and its territories.
2. Out-of-Scope Requests: If a user asks a general question unrelated to weather (e.g., math, coding, general trivia, recipes), politely decline by stating: "I can only assist with weather-related inquiries for locations within the United States."
3. Foreign Locations: The NWS API only covers US cities and territories. If a location is outside the US (e.g., Paris, Tokyo, Toronto), explain that you can only provide weather forecasts for US locations.
"""

weather_agent = Agent(
    name="weather_agent",
    model="gemini-2.5-flash",
    instruction=SYSTEM_PROMPT,
    tools=[
        lookup_coordinates,
        get_weather_forecast,
    ],
)

session_service = InMemorySessionService()

await session_service.create_session(
    app_name="weather_agent",
    user_id="bob",
    session_id="weather-session",
)

runner = Runner(
    app_name="weather_agent",
    agent=weather_agent,
    session_service=session_service,
)

events = runner.run(
    user_id="bob",
    session_id="weather-session",
    new_message=Content(
        role="user",
        parts=[Part(text="What is the weather in Pittsburgh, PA?")]
    ),
)

for event in events:
    if event.is_final_response():
        print(event.content.parts[0].text)


Here is the weather forecast for **Pittsburgh, PA**:

* **This Afternoon:** A chance of showers and thunderstorms. Mostly cloudy, with a high near 85°F. Southwest wind around 8 mph. Chance of precipitation is 50%.
* **Tonight:** Showers and thunderstorms likely, especially after 3 AM. Mostly cloudy with a low around 69°F. Southwest wind 3 to 7 mph. Chance of precipitation is 80%.
* **Saturday:** Showers and thunderstorms likely before 10 AM, with a lingering chance through late afternoon. High near 85°F with southwest winds 6 to 13 mph. Chance of precipitation is 70%.
* **Saturday Night:** A chance of showers and thunderstorms before midnight, then becoming partly cloudy. Low around 69°F.
* **Sunday:** Mostly sunny and pleasant with a high near 87°F. West wind 3 to 7 mph.


Test code

In [138]:
questions = [
    "What is the weather in Pittsburgh, PA?",
    "How about Philadelphia?",
    "Will I need an umbrella today?",
    "What about tomorrow's forecast?"
]

for question in questions:
    print("=" * 80)
    print("USER:", question)

    events = runner.run(
        user_id="bob",
        session_id="weather-session",
        new_message=Content(
            role="user",
            parts=[Part(text=question)]
        ),
    )

    for event in events:
        if event.is_final_response():
            print("AGENT:", event.content.parts[0].text)

USER: What is the weather in Pittsburgh, PA?
AGENT: Here is the forecast for **Pittsburgh, PA**:

* **This Afternoon:** A chance of showers and thunderstorms. Mostly cloudy, with a high near 85°F. Southwest wind around 8 mph (50% chance of rain).
* **Tonight:** Showers and thunderstorms likely, with a low around 69°F. Southwest wind 3 to 7 mph (80% chance of rain).
* **Saturday:** Showers and thunderstorms likely, mainly before 10 AM, with scattered showers continuing into the afternoon. High near 85°F with a 70% chance of rain.
* **Saturday Night:** A lingering chance of showers/thunderstorms before midnight, then becoming partly cloudy with a low around 69°F.
* **Sunday:** Mostly sunny with a high near 87°F and a west wind around 3 to 7 mph.
USER: How about Philadelphia?
AGENT: Here is the weather forecast for **Philadelphia, PA**:

* **Today:** Patchy fog early, giving way to sunny skies with a chance of showers and thunderstorms. High near 92°F (heat index values as high as 102°F).

In [129]:
from vertexai import agent_engines

app = agent_engines.AdkApp(
    agent=weather_agent
)

In [134]:
from vertexai import types
from vertexai import agent_engines


STAGING_BUCKET = "gs://challenge-5-bc"

remote_agent = client.agent_engines.create(
    agent=app,
    config={
        "requirements": ["google-cloud-aiplatform[agent_engines,adk]",
                         "pandas",
                         "requests"],
        "staging_bucket": STAGING_BUCKET,
        "env_vars": {
            "GOOGLE_MAPS_API_KEY": GOOGLE_MAPS_API_KEY,
        },
    },
)

INFO:vertexai_genai.agentengines:Identified the following requirements: {'cloudpickle': '3.1.2', 'google-cloud-aiplatform': '1.163.0', 'pydantic': '2.13.4'}
INFO:vertexai_genai.agentengines:The following requirements are appended: {'pydantic==2.13.4', 'cloudpickle==3.1.2'}
INFO:vertexai_genai.agentengines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'pandas', 'requests', 'pydantic==2.13.4', 'cloudpickle==3.1.2']
INFO:vertexai_genai.agentengines:Using bucket challenge-5-bc
INFO:vertexai_genai.agentengines:Wrote to gs://challenge-5-bc/agent_engine/agent_engine.pkl
INFO:vertexai_genai.agentengines:Writing to gs://challenge-5-bc/agent_engine/requirements.txt
INFO:vertexai_genai.agentengines:Creating in-memory tarfile of extra_packages
INFO:vertexai_genai.agentengines:Writing to gs://challenge-5-bc/agent_engine/dependencies.tar.gz
INFO:vertexai_genai.agentengines:Using agent framework: google-adk
INFO:vertexai_genai.agentengines:View progress and logs at ht

In [135]:
remote_agent = client.agent_engines.get(
    name="projects/1054073855712/locations/us-central1/reasoningEngines/7216043685803196416"
)

In [136]:
import asyncio

async def test_agent():
    async for event in remote_agent.async_stream_query(
        user_id="bob",
        message="What is the forecast in Dallas, TX today?"
    ):
        print(event)

await test_agent()

{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'function_call': {'id': 'adk-c37bc210-5445-4adb-975c-9a734b494233', 'args': {'city': 'Dallas', 'state': 'TX'}, 'name': 'lookup_coordinates'}, 'thought_signature': 'CuoCAY89a18gHFblYjzg25bssfOgWSQJo9vkYcePicy8cEWGY-1YzsY1zeKPr2g_MNWBc3DB0iAEvT2HW2uCLeH-nxawI1jFApr011M-iwADImEnNh5Mk7br8jFDxSXKPTttCoY06UgQh2nRMEBm8F7ArZxAnyR1iGq332Li43988qiNaqx40ibFRXmULOwGNK2sfC-aVIqR-O03kPjd3KhjmduAvpyelJ9xh29FSbKTDquKGFy698fHXhYO32SZVD5keHyOu2S6SepCHvKwNeZUCze9ljXeD1VZHKDRKVYKNkwmVnUVCmyzX1BzFU0-Eo0cM91TKBIgFLKVI7Ej-h99NedwX72QurhQjuXtRN3Lo4wpe4JN6mlbUTPCo2Kw_N7FtNB_c95x0abUtUIYrupufrtP08GilHX97ImfMw1l_I35q4s3uDvEAv1jDayJJC0r4QswfIgub67-hIR-kYtlGQyaC7v-LWaT52w='}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 7, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 7}], 'prompt_token_count': 272, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 272}], 'thoughts_token_count': 